# Lab 2: From Case Notes to Vectors

## This lab has 4 steps.

1. Create the retrieval chunk table.
2. Chunk the source cases without duplicating prior work.
3. Make one explicit embedding call.
4. Embed only unfinished chunks.

## Lab Goals

Turn the source narratives into traceable text chunks and stored vectors that semantic search can compare.

## The challenge

We have a model connection, but no searchable vectors yet. Now we build the
bridge from case descriptions to embeddings: text is split into manageable
pieces, each piece retains its source case ID, and the model turns its meaning
into a vector stored beside it.

![A source case is split into overlapping chunks, each sent to the same embedding model and stored with its case ID, text, and 1536-dimensional vector.](../img/chunk-and-embed.png)

Data preparation preserves provenance: a retrieved vector leads back to its text and source case. Overlap is a percentage of chunk size; this lab uses 800 characters and 10 percent overlap (80 characters).

Our source cases are short, so many will fit in one chunk. This lets us
discuss when chunking is useful and how chunk size affects retrieval.
Larger chunks retain context; smaller chunks can
focus a match but lose surrounding detail.

## Starting point

Complete notebook 1, keep the MSSQL connection on `Healthcare`, and run cells
individually. The first smoke call and the batch call are explicitly labeled
because they incur inference charges. Existing successful embeddings are
preserved on a rerun.

The cases are synthetic, not medical advice. See the [lab guide](README.md)
for cost, recovery, and prepared-environment requirements.


## Step 00: Verify this lab is ready to run

A fresh VM should already have the required database setup. Check that state before creating or calling anything.
This read-only checkpoint reports one ordered Result column and makes no
billed model calls. Every required check must say **Yes**.

A **No** or **Skipped** result produces controlled error 51000 after the
grid. Follow its diagnostic and rerun the checkpoint; the script does not
provision or repair resources for you.


In [ ]:
SET NOCOUNT ON;
DECLARE @Checks TABLE (
    Sort INT IDENTITY PRIMARY KEY, [Check] NVARCHAR(100),
    Passed BIT NOT NULL, Skipped BIT NOT NULL DEFAULT 0, Details NVARCHAR(2048)
);
DECLARE @DatabaseReady BIT = CASE WHEN DB_NAME() = N'Healthcare' THEN 1 ELSE 0 END;
DECLARE @Passed BIT;
INSERT INTO @Checks ([Check], Passed, Details)
VALUES (N'Workshop database', @DatabaseReady, N'Connect the MSSQL kernel to Healthcare, not master.');

SET @Passed = 0;
IF @DatabaseReady = 1 AND (OBJECT_ID(N'dbo.Cases', N'U') IS NOT NULL)
    EXEC sys.sp_executesql
        N'SELECT @Ok = CASE WHEN (SELECT COUNT(*) FROM dbo.Cases) = 207 THEN 1 ELSE 0 END;',
        N'@Ok BIT OUTPUT', @Ok = @Passed OUTPUT;
INSERT INTO @Checks ([Check], Passed, Skipped, Details)
VALUES (N'Seeded cases', @Passed,
    CASE WHEN @DatabaseReady = 0 OR NOT (OBJECT_ID(N'dbo.Cases', N'U') IS NOT NULL) THEN 1 ELSE 0 END,
    N'Complete prework: original Labs 02 and 03, with all 207 synthetic cases.');

SET @Passed = 0;
IF @DatabaseReady = 1 AND (1 = 1)
    EXEC sys.sp_executesql
        N'SELECT @Ok = CASE WHEN EXISTS (SELECT 1 FROM sys.external_models m
    JOIN sys.database_scoped_credentials c ON c.credential_id = m.credential_id
    WHERE m.name = N''embedding_openai_text3_small''
      AND c.credential_identity = N''Managed Identity'') THEN 1 ELSE 0 END;',
        N'@Ok BIT OUTPUT', @Ok = @Passed OUTPUT;
INSERT INTO @Checks ([Check], Passed, Skipped, Details)
VALUES (N'External embedding model', @Passed,
    CASE WHEN @DatabaseReady = 0 OR NOT (1 = 1) THEN 1 ELSE 0 END,
    N'Complete retrieval Lab 01 and inspect the model/credential association.');

SELECT CONCAT([Check], N': ', CASE
    WHEN Skipped = 1 THEN CONCAT(N'Skipped - ', Details)
    WHEN Passed = 1 THEN N'Yes'
    ELSE CONCAT(N'No - ', Details)
END) AS Result
FROM @Checks ORDER BY Sort;
IF EXISTS (SELECT 1 FROM @Checks WHERE Passed = 0)
    THROW 51000, 'Resolve the failed or skipped required checks above.', 1;


### Expected Output
Estimated time: 1-5 seconds.

| Result |
|---|
| Workshop database: Yes |
| Seeded cases: Yes |
| External embedding model: Yes |


## Step 1: Create the retrieval chunk table

Create a table for the prepared text and its vector. `CaseId` links
each chunk back to its source; `Chunk` keeps the evidence readable; and the
nullable `VECTOR(1536)` column gives us somewhere to store the chosen model's
output. The clustered primary key also supports the later index exercise.

The table belongs to this track. We do not reuse or reset the original
workshop's complaint/symptom embedding tables, and a rerun does not clear
existing chunks or vectors.

**Learn more:** [The vector data type](https://learn.microsoft.com/en-us/sql/t-sql/data-types/vector-data-type?view=sql-server-ver17).


In [ ]:
IF OBJECT_ID(N'dbo.RetrievalChunks', N'U') IS NULL
    CREATE TABLE dbo.RetrievalChunks (
        Id INT IDENTITY(1,1) CONSTRAINT PK_RetrievalChunks PRIMARY KEY CLUSTERED,
        CaseId INT NOT NULL REFERENCES dbo.Cases(Id),
        Chunk NVARCHAR(MAX) NOT NULL,
        Embedding VECTOR(1536) NULL
    );


### Expected result
Estimated time: 1-5 seconds.

`dbo.RetrievalChunks` exists; rerunning does not clear text or embeddings.


### Inspect the result

Inspect the table shape before loading it. The column metadata should
identify `Embedding` as the declared `vector` type, with 1536 dimensions and
nullable storage.

The dimension is a compatibility contract with the selected model, not a
measure of clinical confidence. If the check fails, fix the schema before
spending inference calls on data that cannot be stored.


In [ ]:
SELECT name, TYPE_NAME(user_type_id) AS DeclaredType, vector_dimensions, is_nullable
FROM sys.columns WHERE object_id = OBJECT_ID(N'dbo.RetrievalChunks');
IF NOT EXISTS (SELECT 1 FROM sys.columns
    WHERE object_id = OBJECT_ID(N'dbo.RetrievalChunks') AND name = N'Embedding'
      AND TYPE_NAME(user_type_id) = N'vector' AND vector_dimensions = 1536 AND vector_base_type = 0)
    THROW 51000, 'RetrievalChunks requires a float32 VECTOR(1536) Embedding column.', 1;


### Expected result
Estimated time: 1-5 seconds.

Embedding reports declared type `vector`, dimensions `1536`, and `is_nullable = 1`.


## Step 2: Chunk the source cases without duplicating prior work

Combine each complaint and symptom description, then split the text
into fixed-size chunks. Each output retains its source `CaseId`. The
`NOT EXISTS` guard skips cases already processed, so a normal rerun does not
duplicate chunks or discard embeddings.

**Pause and consider:** If a relevant sentence sits on a chunk boundary,
what context might be lost? What is the cost of keeping more overlap?

Here `chunk_size = 800` counts characters; `overlap = 10` means **10 percent**
of that size, or 80 characters. Many short cases fit in a single chunk.
This is an append-once lab workflow, not synchronization of later source edits.

**Learn more:** [AI_GENERATE_CHUNKS](https://learn.microsoft.com/en-us/sql/t-sql/functions/ai-generate-chunks-transact-sql?view=sql-server-ver17).


In [ ]:
INSERT INTO dbo.RetrievalChunks (CaseId, Chunk)
SELECT c.Id, ch.chunk
FROM dbo.Cases c
CROSS APPLY AI_GENERATE_CHUNKS(
    source = CONCAT(c.Complaint, N'. ', c.Symptoms),
    chunk_type = FIXED, chunk_size = 800, overlap = 10
) ch
WHERE NOT EXISTS (SELECT 1 FROM dbo.RetrievalChunks existing WHERE existing.CaseId = c.Id);


### Expected result
Estimated time: 1-5 seconds.

Every source case has at least one chunk. Rerunning inserts no duplicate cases; it does not synchronize later source edits.


### Inspect the result

Check coverage before calling a model: every source case should have
at least one chunk, and `MissingCases` must be zero. Inspect a few rows to
see the source IDs, chunk lengths, and text that will actually be embedded.

On a fresh run the embeddings are NULL. If you are resuming prior work,
already generated embeddings should still be present.


In [ ]:
SELECT COUNT(*) AS Chunks, COUNT(DISTINCT CaseId) AS CasesCovered,
    COUNT(Embedding) AS EmbeddedChunks FROM dbo.RetrievalChunks;
SELECT COUNT(*) AS MissingCases FROM dbo.Cases c
WHERE NOT EXISTS (SELECT 1 FROM dbo.RetrievalChunks ch WHERE ch.CaseId = c.Id);
SELECT TOP (5) Id, CaseId, LEN(Chunk) AS Characters, Chunk
FROM dbo.RetrievalChunks ORDER BY Id;


### Expected result
Estimated time: 1-5 seconds.

`CasesCovered = 207`, `MissingCases = 0`, at least 207 chunks. Previously generated embeddings are preserved.


## Step 3: Make one explicit embedding call

Make one small, explicit inference request using the model registered
in notebook 1. We will show its dimension and only a short numeric preview:
reading all the coordinates would not explain the text's meaning.

The response tests the real endpoint, identity permissions, and deployment,
which metadata alone could not prove. Expect 1536 dimensions; resolve a NULL
result or service error before attempting the batch.

> **Inference cost:** One billed embedding request per execution. Run this
> cell deliberately, not as part of repeated Run All attempts.

**Learn more:** [AI_GENERATE_EMBEDDINGS](https://learn.microsoft.com/en-us/sql/t-sql/functions/ai-generate-embeddings-transact-sql?view=sql-server-ver17) and [The vector data type](https://learn.microsoft.com/en-us/sql/t-sql/data-types/vector-data-type?view=sql-server-ver17).


In [ ]:
DECLARE @Embedding VECTOR(1536) = AI_GENERATE_EMBEDDINGS(
    N'crushing chest pain' USE MODEL embedding_openai_text3_small
);
IF @Embedding IS NULL
    THROW 51000, 'The embedding model returned NULL. Check the endpoint, identity permissions and quota.', 1;
SELECT VECTORPROPERTY(@Embedding, 'Dimensions') AS Dimensions,
    LEFT(CAST(@Embedding AS NVARCHAR(MAX)), 160) AS VectorPreview;


### Expected result
Estimated time: 1-15 seconds.

`Dimensions = 1536` and a numeric-array preview. Authentication or model failures must be resolved before the batch.


## Step 4: Embed only unfinished chunks

Now fill the pending vectors. The loop selects two unfinished chunks
per statement, calls the same embedding model for their text, and saves the
results beside the chunks. Successful earlier batches remain committed if a
later batch fails.

Watch the progress messages. A visible model error stops the run; resolve the
endpoint, permission, quota, or service issue before rerunning. Only NULL
embeddings are selected, so the rerun resumes rather than starting over.

> **Inference cost:** Up to one billed embedding request per pending chunk.
> Do not clear successful vectors to repeat the demonstration. This lab does
> not hide service failures with automatic retries or cached results.

**Learn more:** [AI_GENERATE_EMBEDDINGS](https://learn.microsoft.com/en-us/sql/t-sql/functions/ai-generate-embeddings-transact-sql?view=sql-server-ver17).


In [ ]:
SET NOCOUNT ON;
DECLARE @Batch TABLE (Id INT PRIMARY KEY);
DECLARE @Done INT = 0;
WHILE EXISTS (SELECT 1 FROM dbo.RetrievalChunks WHERE Embedding IS NULL)
BEGIN
    DELETE FROM @Batch;
    INSERT INTO @Batch (Id)
    SELECT TOP (2) Id FROM dbo.RetrievalChunks WHERE Embedding IS NULL ORDER BY Id;

    UPDATE ch SET Embedding = AI_GENERATE_EMBEDDINGS(
        ch.Chunk USE MODEL embedding_openai_text3_small
    )
    FROM dbo.RetrievalChunks ch JOIN @Batch b ON b.Id = ch.Id
    WHERE ch.Embedding IS NULL;
    SET @Done += @@ROWCOUNT;

    IF EXISTS (SELECT 1 FROM dbo.RetrievalChunks ch JOIN @Batch b ON b.Id = ch.Id
        WHERE ch.Embedding IS NULL)
        THROW 51000, 'A model call returned NULL. Fix model access or quota, then rerun to resume.', 1;
    RAISERROR('Embedded another batch; %d chunks processed in this run.', 0, 1, @Done) WITH NOWAIT;
END;


### Expected result
Estimated time: 30-240 seconds; quota dependent.

Each successful update persists. A completed rerun makes no chunk embedding calls.


### Inspect the result

Check that the model work is complete before moving to search. The
pending count must be zero, and the number of populated embeddings must
match the chunk count.

The index exercise needs at least 100 populated vectors; the complete
207-case dataset provides enough rows. A successful model smoke call is not
a substitute for this coverage check.

**Reflect:** Why would mixing stored embeddings from one model with query
embeddings from another undermine the comparison?


In [ ]:
SELECT COUNT(*) AS Chunks, COUNT(Embedding) AS Embedded,
    COUNT(*) - COUNT(Embedding) AS Pending FROM dbo.RetrievalChunks;
IF EXISTS (SELECT 1 FROM dbo.RetrievalChunks WHERE Embedding IS NULL)
    THROW 51000, 'Embeddings are incomplete. Resume the previous cell.', 1;
IF (SELECT COUNT(Embedding) FROM dbo.RetrievalChunks) < 100
    THROW 51000, 'The vector index needs at least 100 populated vectors. Check case/chunk coverage.', 1;


### Expected result
Estimated time: 1-5 seconds.

`Pending = 0`; every chunk has a vector, with at least 207 on the complete seed dataset.


## Final Step: Validate this lab is complete

Before moving on, verify the persistent handoffs this notebook owns or depends on.
This read-only checkpoint reports one ordered Result column and makes no
billed model calls. Every required check must say **Yes**.

A **No** or **Skipped** result produces controlled error 51000 after the
grid. Follow its diagnostic and rerun the checkpoint; the script does not
provision or repair resources for you.

This checks persisted state, not live connectivity, search relevance, or index use. Those require inspecting the explicit exercises above.


In [ ]:
SET NOCOUNT ON;
DECLARE @Checks TABLE (
    Sort INT IDENTITY PRIMARY KEY, [Check] NVARCHAR(100),
    Passed BIT NOT NULL, Skipped BIT NOT NULL DEFAULT 0, Details NVARCHAR(2048)
);
DECLARE @DatabaseReady BIT = CASE WHEN DB_NAME() = N'Healthcare' THEN 1 ELSE 0 END;
DECLARE @Passed BIT;
INSERT INTO @Checks ([Check], Passed, Details)
VALUES (N'Workshop database', @DatabaseReady, N'Connect the MSSQL kernel to Healthcare, not master.');

SET @Passed = 0;
IF @DatabaseReady = 1 AND (OBJECT_ID(N'dbo.RetrievalChunks', N'U') IS NOT NULL AND OBJECT_ID(N'dbo.Cases', N'U') IS NOT NULL)
    EXEC sys.sp_executesql
        N'SELECT @Ok = CASE WHEN EXISTS (SELECT 1 FROM dbo.RetrievalChunks)
    AND NOT EXISTS (SELECT 1 FROM dbo.Cases c
      WHERE NOT EXISTS (SELECT 1 FROM dbo.RetrievalChunks ch WHERE ch.CaseId = c.Id)) THEN 1 ELSE 0 END;',
        N'@Ok BIT OUTPUT', @Ok = @Passed OUTPUT;
INSERT INTO @Checks ([Check], Passed, Skipped, Details)
VALUES (N'Chunks cover every case', @Passed,
    CASE WHEN @DatabaseReady = 0 OR NOT (OBJECT_ID(N'dbo.RetrievalChunks', N'U') IS NOT NULL AND OBJECT_ID(N'dbo.Cases', N'U') IS NOT NULL) THEN 1 ELSE 0 END,
    N'Run retrieval Lab 02; every source case must have at least one chunk.');

SET @Passed = 0;
IF @DatabaseReady = 1 AND (OBJECT_ID(N'dbo.RetrievalChunks', N'U') IS NOT NULL)
    EXEC sys.sp_executesql
        N'SELECT @Ok = CASE WHEN (SELECT COUNT(*) FROM dbo.RetrievalChunks WHERE Embedding IS NOT NULL) >= 100
    AND NOT EXISTS (SELECT 1 FROM dbo.RetrievalChunks WHERE Embedding IS NULL) THEN 1 ELSE 0 END;',
        N'@Ok BIT OUTPUT', @Ok = @Passed OUTPUT;
INSERT INTO @Checks ([Check], Passed, Skipped, Details)
VALUES (N'Complete embeddings', @Passed,
    CASE WHEN @DatabaseReady = 0 OR NOT (OBJECT_ID(N'dbo.RetrievalChunks', N'U') IS NOT NULL) THEN 1 ELSE 0 END,
    N'Resume retrieval Lab 02; all chunks need vectors, with at least 100 for the vector index.');

SELECT CONCAT([Check], N': ', CASE
    WHEN Skipped = 1 THEN CONCAT(N'Skipped - ', Details)
    WHEN Passed = 1 THEN N'Yes'
    ELSE CONCAT(N'No - ', Details)
END) AS Result
FROM @Checks ORDER BY Sort;
IF EXISTS (SELECT 1 FROM @Checks WHERE Passed = 0)
    THROW 51000, 'Resolve the failed or skipped required checks above.', 1;


### Expected Output
Estimated time: 1-5 seconds.

| Result |
|---|
| Workshop database: Yes |
| Chunks cover every case: Yes |
| Complete embeddings: Yes |


Previous: [Notebook 1](01-beyond-keywords.ipynb) | [Session guide](README.md) | Next: [Notebook 3](03-search-by-meaning.ipynb)
